# 19 - Python Best Practices, Formatting & Internals


## 1. Advanced F-String Formatting (Alignment, Fill, Precision & Scientific Notation)

Python f-string ifadeleri, süslü parantez içinde iki nokta (`:`) ardından gelen biçimlendirme belirticileri (format specifiers) ile metin hizalama, dolgu karakteri ekleme, basamak yuvarlama ve bilimsel gösterim dönüşümlerini tek satırda yapmayı sağlar.

**Syntax**

```Python
f'{text:_<20}'
f'{text:_>20}'
f'{text:_^20}'
f'{number:,.2f}'
f'{ratio:.2%}'
f'{big_num:.0e}'
```

**Biçimlendirme Belirticileri ve Kuralları**

* **Alan Genişliği ve Hizalama (`<`, `>`, `^`):** `:20` ifadesi değişken için en az 20 karakterlik yer ayırır; `<` sola, `>` sağa ve `^` ortaya hizalama yapar. Hizalama okunun önüne yazılan karakter (örneğin `_^20`) boş kalan alanları o karakterle doldurur.
* **Ondalık Hassasiyet ve Binlik Ayırıcı (`,.2f`):** `.2f` kayan noktalı sayıyı 2 ondalık basamağa yuvarlar; önüne eklenen virgül (`,`) binlik basamak ayracı ekler.
* **Yüzde ve Bilimsel Gösterim (`.2%`, `.0e`):** `%` belirtici sayıyı 100 ile çarparak yüzde formatında yazdırır; `e` belirtici ise sayıyı üstel/bilimsel gösterime (`1e+06`) dönüştürür.


In [ ]:
text: str = 'sample'
print(f'{text:<20}: left')
print(f'{text:>20}: right')
print(f'{text:^20}: center')
print(f'{text:_<20}')
print(f'{text:_>20}')
print(f'{text:_^20}')

number: float = 10000.123456
ratio: float = 0.096
large_int: int = 1_000_000

print(f'{number:,.2f}')
print(f'{ratio:.2%}')
print(f'{large_int:.0e}')


## 2. Tuples & Type Hinting (`tuple[T, ...]`)

Demetler (tuples) sabit uzunluklu ve konuma duyarlı yapılar olarak ele alındığı için, tip ek açıklamalarında (type hinting) her bir elemanın tipi ayrı ayrı belirtilir; ancak değişken uzunluklu homojen veya çok tipli demetler için ellipsis (`...`) ve birleşim (`|`) operatörü kullanılır.

**Syntax**

```Python
fixed_pair: tuple[int, str] = (12, 'hello')
variable_ints: tuple[int, ...] = (1, 2, 3, 4)
mixed_items: tuple[int | str, ...] = (12, 'hello', 99)
```

**Kullanım Kuralları ve Dikkat Edilmesi Gerekenler**

* **Sabit Uzunluk (`tuple[int, str]`):** Köşeli parantez içine yazılan tip sayısı, demetin tam olarak o uzunlukta ve o sırayla eleman içermesi gerektiğini belirtir.
* **Değişken Uzunluk (`tuple[int, ...]`):** İkinci sıraya konulan `...` (ellipsis), baştaki tipin sıfır veya daha fazla kez tekrarlanabileceğini statik tip denetleyicilere bildirir.
* **Çoklu Tip Desteği (`|`):** `tuple[int | str, ...]` ifadesi, demet içinde sınırsız sayıda `int` veya `str` öğesinin karışık halde bulunabileceğini tanımlar.


In [ ]:
fixed_tuple: tuple[int, str] = (12, 'hello')
unlimited_ints: tuple[int, ...] = (1, 2, 3, 4, 5)
mixed_tuple: tuple[int | str, ...] = (12, 'hello', 34, 'world')

print(fixed_tuple)
print(unlimited_ints)
print(mixed_tuple)


## 3. Flattening Nested Lists (List Comprehension & Recursion)

İç içe geçmiş çok boyutlu listeleri tek boyutlu düz bir listeye indirmek (flattening) için 2 boyutlu yapılarda list comprehension, derinliği belirsiz çok boyutlu yapılarda ise özyinelemeli (recursive) fonksiyonlar kullanılır.

**Syntax**

```Python
flat_2d: list[str] = [name for group in people for name in group]
```

**Yaklaşımlar ve Kritik Detaylar**

* **2 Boyutlu List Comprehension:** Dış döngü solda, iç döngü sağda yazılır (`[item for sublist in matrix for item in sublist]`) ve tek satırda hızlı düzleştirme sağlar.
* **Keyfi Derinlikte Özyineleme (`Iterable` Kontrolü):** `collections.abc.Iterable` ile öğenin yinelenebilir olup olmadığı kontrol edilir; yinelenebilir ise özyinelemeli olarak `extend()`, değilse `append()` çağrılır.
* **`str` ve `bytes` İstisnası:** Python'da `str` de bir `Iterable` olduğu için, metinlerin karakter karakter parçalanmasını önlemek amacıyla `not isinstance(item, (str, bytes))` koşulu mutlaka eklenmelidir.


In [ ]:
from collections.abc import Iterable
from typing import Any

people_2d: list[list[str]] = [['Mario', 'Luigi'], ['James', 'Bob']]
flat_people: list[str] = [name for group in people_2d for name in group]
print('2D Comprehension:', flat_people)


def flatten(items: Iterable[Any]) -> list[Any]:
    result: list[Any] = []
    for item in items:
        if isinstance(item, Iterable) and not isinstance(item, (str, bytes)):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result


nested_data: list[Any] = ['James', 'Bob', [1, 2, [True, False], 3, 4], 'End']
print('Recursive Flatten:', flatten(nested_data))


## 4. The Ellipsis Object (`...`)

`...` (Ellipsis), tıpkı `None` gibi Python'ın yerleşik (built-in) tekil (singleton) sabit nesnelerinden biridir ve kod içinde henüz yazılmamış bloklar için yer tutucu, değişken uzunluklu tip belirteci ve çok boyutlu dilimleme operatörü olarak görev yapar.

**Syntax**

```python
def unimplemented_task() -> None:
    ...

items: tuple[int, ...] = (1, 2, 3)
```

**Kullanım Alanları**

* **Tekil Yerleşik Nesne (`Ellipsis`):** `... is Ellipsis` ifadesi her zaman `True` döner.
* **Kod Yer Tutucu (Placeholder):** Boş bırakılamayan fonksiyon, sınıf veya `if` gövdelerinde `pass` deyimine alternatif olarak sözdizimi hatasını önler (genel kullanımda `pass` daha yaygındır, soyut/stub dosyalarda `...` tercih edilir).
* **Tip İpuçları ve Çok Boyutlu Dilimleme:** `tuple[T, ...]` tanımlarında belirsiz uzunluğu ifade eder; ayrıca NumPy gibi kütüphanelerde çok boyutlu dizilerin ara eksenlerini atlayarak dilimlemek (`arr[..., 0]`) için kullanılır.


In [ ]:
def do_something() -> None:
    ...


do_something()
print('Literal:', ...)
print('Identity check:', ... is Ellipsis)

numbers: tuple[int, ...] = (10, 20, 30, 40)
print('Tuple:', numbers)


## 5. Static Code Analysis & Linting (`pylint`)

Linter araçları (örneğin `pylint`), kodu çalıştırmadan statik analiz yaparak sözdizimi hatalarını, kullanılmayan import/değişkenleri, PEP 8 stil ihlallerini ve yeniden düzenleme (refactoring) gerektiren yapıları tespit eder.

**Syntax**

```bash
pylint main.py
```

**`pylint` Mesaj Kategorileri**

* **`C` (Convention):** PEP 8 adlandırma, girinti veya docstring gibi kodlama standartları ihlalleri.
* **`R` (Refactor):** Çok az genel metoda sahip sınıflar veya karmaşık bloklar gibi yapısal iyileştirme önerileri.
* **`W` (Warning):** Kullanılmayan değişkenler veya gereksiz `pass` ifadeleri gibi stil/programlama uyarıları.
* **`E` (Error):** Tanımsız değişken kullanımı gibi çalışma zamanında muhtemel hataya yol açacak ciddi sorunlar.
* **`F` (Fatal):** Analizin devam etmesini engelleyen kritik ayrıştırma (parse) hataları.


In [ ]:
class Fruit:
    def __init__(self, name: str) -> None:
        self.name = name

    def explode(self) -> str:
        return f'{self.name} exploded!'

    def describe(self) -> str:
        return f'Fruit: {self.name}'


apple = Fruit('Apple')
print(apple.describe())
print(apple.explode())


## 6. Suppressing Linter Rules (`# noqa`)

`# noqa` (*No Quality Assurance*), linter ve stil denetleyicilerine (Flake8, Ruff, Pylint vb.) ilgili satırdaki kural ihlalinin bilinçli yapıldığını bildirerek uyarıların bastırılmasını sağlayan satır içi yorum etiketidir.

**Syntax**

```python
import math, random  # noqa: E401
greet = lambda text: print(text)  # noqa: E731
```

**En İyi Uygulamalar (Best Practices)**

* **Kural Kodu Belirtmek (`# noqa: E401`):** Tek başına `# noqa` yazmak o satırdaki *tüm* uyarıları susturarak gerçek hataları gizleyebilir. Bunun yerine yalnızca bilerek ihlal edilen kuralın kodunu (`E401`, `E731` vb.) eklemek diğer denetimleri aktif tutar.


In [ ]:
import math, random  # noqa: E401

greet = lambda text: f'Hello, {text}'  # noqa: E731

print(round(math.sqrt(10), 3))
print(round(random.random(), 3))
print(greet('Python'))


## 7. Efficient String Concatenation (`+` vs. `''.join()`)

Python'da `str` nesneleri değiştirilemez (*immutable*) olduğu için, döngü içinde `+=` ile yapılan her birleştirme işleminde bellekte yepyeni bir string nesnesi tahsis edilir ve önceki içerik kopyalanır.

**Syntax**

```python
implicit_text = 'Hello ' 'World'
joined_text = ''.join(parts_list)
```

**Performans ve Tasarım Kuralları**

* **Örtük Birleştirme (Implicit Concatenation):** Yan yana yazılan string sabitleri (`'Hello ' 'World'`) derleme zamanında otomatik birleştirilir.
* **Döngü İçi `+=` Maliyeti:** Çok sayıda parça birleştirilirken `+=` kullanmak her adımda yeni nesne oluşturduğu için bellek ve zaman karmaşıklığını ($O(n^2)$) artırır.
* **`list.append()` + `str.join()` Modeli:** Parçaları önce bir listede toplayıp sonunda tek bir `''.join(items)` çağrısıyla birleştirmek toplam uzunluğu önceden hesaplayarak tek seferde bellek ayırır ve büyük veri setlerinde çok daha hızlıdır.


In [ ]:
import timeit

implicit_greeting: str = 'Hello ' 'World'
print(implicit_greeting)


def concat_with_plus() -> str:
    result: str = ''
    for _ in range(1000):
        result += 'string'
    return result


def concat_with_join() -> str:
    parts: list[str] = []
    for _ in range(1000):
        parts.append('string')
    return ''.join(parts)


assert len(concat_with_plus()) == len(concat_with_join())

time_plus: float = timeit.timeit(concat_with_plus, number=20_000)
time_join: float = timeit.timeit(concat_with_join, number=20_000)

print(f'Plus (+=): {time_plus:.4f}s')
print(f'List join: {time_join:.4f}s')


## 8. Backslashes (`\`): Escape Sequences & Line Continuation

Ters eğik çizgi (`\`), string ifadeler içinde özel kontrol karakterlerini ve kaçış dizilerini (escape sequences) temsil ederken, kod satırlarının sonunda ifadeyi bir alt satıra devam ettirmek (explicit line continuation) için kullanılır.

**Syntax**

```python
escaped = 'It\'s a quote\nNext line\tTabbed'
hex_char = '\x48'
oct_char = '\110'
```

**Temel Kaçış Dizileri ve Satır Devamı**

* **Tırnak ve Boşluk Kontrolleri:** `\'` (tek tırnak), `\n` (yeni satır), `\t` (yatay sekme), `\b` (backspace), `\r` (carriage return — imleci satır başına döndürür) ve `\v` (dikey sekme).
* **Sayısal Karakter Kodları:** `\xHH` onaltılık (hexadecimal) değeri, `\ooo` sekizlik (octal) değeri ASCII/Unicode karakterine dönüştürür (örneğin `\x48` ve `\110` `'H'` harfini üretir).
* **Açık Satır Devamı (Line Continuation):** Satır sonuna konulan `\` ifadeyi sonraki satıra bağlar; ancak PEP 8 kapsamında matematiksel veya mantıksal ifadelerde `\` yerine parantez `(...)` içinde örtük satır bölme tercih edilir.


In [ ]:
print('This is a \'single quote\' inside single quotes')
print('Line 1\nLine 2\tTabbed')
print('Hex 48:', '\x48', '| Octal 110:', '\110')
print('Backspace demo:', 'text\b')

continued_text: str = (
    'First part of the sentence '
    'second part on the next line.'
)
print(continued_text)

total_value: int = 10 + 12 * 15 \
    + 10 - 10
print('Calculated total:', total_value)


## 9. Common Uses of the Underscore (`_`)

Alt çizgi (`_`), Python'da hem sözdizimsel bir ayırıcı hem de geliştiriciler arasında yerleşik bir adlandırma konvansiyonu olarak altı farklı rolde kullanılır.

**Syntax**

```python
billion = 1_000_000_000
first, *_, last = (1, 2, 3, 4, 5)
for _ in range(3):
    ...
class_ = 'Python'
```

**Kullanım Senaryoları**

* **Sayısal Basamak Ayırıcı (`1_000_000`):** Büyük tam sayı ve ondalık sayılarda okunabilirliği artırır; çalışma zamanında değeri değiştirmez.
* **Değer Yoksayma (Unpacking & Loops):** Tuple açarken kullanılmayacak elemanları (`a, *_, e = values`) veya döngü sayacının kullanılmadığı `for _ in range(n)` yapılarını ifade eder.
* **Ayrılmış Anahtar Kelime Çakışmasını Önleme (`class_`):** Python anahtar kelimeleriyle (`class`, `id`, `type`) çakışan değişken adlarının sonuna tek alt çizgi eklenir.
* **Korumalı Üye Konvansiyonu (`_internal`):** Sınıf içi özellik ve metotların başına eklenen tek alt çizgi, bunların sınıf ve alt sınıflar dışında doğrudan çağrılmaması gerektiğini belirtir (PEP 8 konvansiyonudur, erişimi teknik olarak engellemez).
* **Etkileşimli Konsol (REPL):** Etkileşimli kabukta değerlendirilen son ifadenin sonucunu otomatik saklar.


In [ ]:
billion: int = 1_000_000_000
print('Formatted literal:', billion)

values: tuple[int, int, int, int, int] = (1, 2, 3, 4, 5)
first, *_, last = values
print('First and last:', first, last)

count: int = 0
for _ in range(3):
    count += 1
print('Loop count:', count)

class_: str = 'Advanced Python'


class Item:
    def __init__(self, item_id: int) -> None:
        self._id: int = item_id

    def get_id(self) -> int:
        return self._id


item = Item(12345)
print(class_, '->', item.get_id())


## 10. Python `list` vs. `array.array`

Python'da yerleşik `list` her türlü veri tipini bir arada tutabilen dinamik bir işaretçi dizisiyken, standart kütüphanedeki `array.array` yalnızca tek bir temel C veri tipine (örneğin `'i'` işaretli tam sayı) sahip öğeleri bitişik bellekte sıkıştırılmış olarak saklar.

**Syntax**

```python
from array import array

int_array = array('i', range(10_000))
mixed_list = [1, 'two', 3.0]
```

**Karşılaştırma ve Seçim Kriterleri**

| Özellik | `list` (Yerleşik) | `array.array` (Standart Kütüphane) |
| :--- | :--- | :--- |
| **Veri Tipi Esnekliği** | Heterojen (farklı tipler bir arada) | Homojen (tanımlanan tek tip kodu) |
| **Bellek Tüketimi** | Yüksek (her eleman ayrı `PyObject` referansı) | Düşük (ham C değerleri bitişik bellekte) |
| **Kullanım Senaryosu** | Genel amaçlı koleksiyonlar, küçük/karışık veriler | Büyük tek tipli sayısal diziler (veya bilimsel işlerde NumPy) |

* **Bellek ve Başlangıç Maliyeti:** Çok az elemanlı yapılarda `array` nesnesinin kendi başlık (overhead) maliyeti bulunurken, binlerce sayısal öğe içeren dizilerde `array.array` standart `list` yapısına kıyasla yaklaşık yarı yarıya daha az bellek tüketir.


In [ ]:
import sys
from array import array

compact_array: array[int] = array('i', range(10_000))
standard_list: list[int] = list(range(10_000))

print('Array typecode:', compact_array.typecode)
print('First 5 array items:', compact_array[:5])
print('Array buffer bytes:', compact_array.buffer_info()[1] * compact_array.itemsize)
print('List container bytes:', sys.getsizeof(standard_list))
